# 05 -- Sintesis: catalogo de features y recomendacion de modelado

Quinto y ultimo notebook de la cadena. Lee mecanicamente los artefactos de
02, 03 y 04 -- cada decision de inclusion/exclusion de feature y cada
numero citado en la recomendacion de modelado se calcula en una celda de
codigo de este notebook, no se transcribe de una corrida anterior.

**Cadena**: `01_calidad_y_panel` -> `02_regla_priorizacion` ->
`03_calendario_estacionalidad` -> `04_demanda_y_baseline` -> **`05_sintesis`**.

1. Catalogo de features de calendario.
2. Periodicidad semanal y anual (ACF/STL, referencia de ruido, leave-one-year-out).
3. Precio/promocion y devoluciones (leidas de `diagnostico_features_riesgo.json`, 04).
4. Producto, patron de demanda y racha (`racha_por_par.parquet`, 04).
5. Contrafactual ABC (02).
6. Baseline (04): WAPE por origen/patron y piso exportado.
7. Catalogo consolidado (`features_recomendadas.csv`).
8. Recomendacion de enfoque de modelado.
9. Proximos pasos.

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import (
    PARAMS, DW, registrar_huella, verificar_huella,
    leer_panel_negocio, leer_dim_priorizado, leer_priorizacion,
    leer_efectos_calendario, leer_estacionalidad_sku,
    leer_indices_categoria_sucursal, leer_calendario_habil,
    leer_patron_demanda, leer_baseline_wape,
    leer_racha_por_par, leer_diagnostico_features_riesgo,
    construir_df_razon, indice_dia_semana,
)

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf as acf_func
from statsmodels.tsa.seasonal import STL

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
plt.rcParams['figure.figsize'] = (11, 4)


## 0. Carga y verificacion de huellas

In [2]:
verificar_huella('huella_02.json', ['dim_producto_priorizado.parquet', 'priorizacion_producto_sucursal.parquet'])
verificar_huella('huella_03.json', ['efectos_calendario.csv', 'estacionalidad_sku.parquet'])
verificar_huella('huella_04.json', [
    'patron_demanda_producto_sucursal.parquet', 'baseline_wape.parquet',
    'racha_por_par.parquet', 'diagnostico_features_riesgo.json',
])

dim_producto = leer_dim_priorizado()
prioridad_par = leer_priorizacion()
efectos_calendario = leer_efectos_calendario()
estacionalidad_sku = leer_estacionalidad_sku()
indices_categoria_sucursal = leer_indices_categoria_sucursal()
patron_demanda = leer_patron_demanda()
baseline_wape = leer_baseline_wape()
racha_por_par = leer_racha_por_par()
diagnostico_riesgo = leer_diagnostico_features_riesgo()
fechas_habiles, indice_habil = leer_calendario_habil()

dim_sucursal = pd.read_csv(f'{DW}/dim_sucursal.csv')
SUCURSALES_REALES = [s for s in dim_sucursal['nombre']
                     if s not in ('SIN_SUCURSAL', 'BODEGA_CENTRAL')]  # INV-62: BODEGA_CENTRAL no vende directo
dim_proveedor = pd.read_csv(f'{DW}/dim_proveedor.csv')
fact_inventario_cols = pd.read_csv(f'{DW}/fact_inventario.csv', nrows=0).columns.tolist()
fact_ventas = pd.read_csv(
    f'{DW}/fact_ventas.csv',
    dtype={'codigo_item': str, 'sucursal': 'category', 'codigo_proveedor': 'category'},
    parse_dates=['fecha'],
)
ventas = fact_ventas.loc[(~fact_ventas['es_devolucion']) & (fact_ventas['cantidad'] > 0)].copy()
valor_por_producto = ventas.groupby('codigo_item', observed=True)['valor_total'].sum()
valor_total = valor_por_producto.sum()
prioritarios = set(dim_producto.loc[dim_producto['es_prioritario'], 'codigo_item'])

print('dim_producto:', dim_producto.shape, '| prioridad_par:', prioridad_par.shape)
print('efectos_calendario:', efectos_calendario.shape, '| patron_demanda:', patron_demanda.shape, '| baseline_wape:', baseline_wape.shape)
print('racha_por_par:', racha_por_par.shape, '| diagnostico_riesgo:', diagnostico_riesgo)


Huella verificada OK contra huella_02.json.
Huella verificada OK contra huella_03.json.
Huella verificada OK contra huella_04.json.


dim_producto: (4414, 23) | prioridad_par: (10805, 10)
efectos_calendario: (13, 12) | patron_demanda: (10805, 10) | baseline_wape: (28984, 15)
racha_por_par: (10805, 6) | diagnostico_riesgo: {'demanda_con_descuento': 43.52485841081995, 'demanda_sin_descuento': 46.348509149747656, 'pct_devoluciones_perecedero_refrigerado': 0.18823529411764706, 'pct_catalogo_perecedero_refrigerado': 0.11758042591753512}


## 1. Catalogo de features de calendario

Regla de inclusion aplicada **en codigo** sobre `efectos_calendario.csv`
(03): se incluye un evento si su intervalo de confianza al 95% NO cruza
1.0 -- es el mismo criterio `direccion` que ya calculo 03, no un umbral
nuevo sobre el punto estimado (`|controlado_x - 1| >= 0.05`), que un IC
ancho puede volver poco confiable (se muestra el contraste abajo).

In [3]:
efectos_calendario['decision'] = np.where(
    efectos_calendario['direccion'].str.startswith('sin_efecto_claro'), 'retirar', 'incluir'
)
umbral_punto = PARAMS['UMBRAL_INCLUSION_EFECTO_CALENDARIO']
decision_por_punto = np.where((efectos_calendario['controlado_x'] - 1).abs() >= umbral_punto, 'incluir', 'retirar')
discrepancias = efectos_calendario.loc[efectos_calendario['decision'] != decision_por_punto, ['evento', 'controlado_x', 'ic_bajo', 'ic_alto', 'decision']]

if len(discrepancias):
    print(f'{len(discrepancias)} evento(s) donde el criterio de IC difiere del umbral sobre el punto estimado (>= {umbral_punto}):')
    print(discrepancias.to_string(index=False))
    print('Se usa el criterio de IC (mas conservador, evita incluir features basadas en ruido de muestra).')
else:
    print('Ambos criterios coinciden en todos los eventos -- no hay discrepancia que resolver.')


1 evento(s) donde el criterio de IC difiere del umbral sobre el punto estimado (>= 0.05):
            evento  controlado_x  ic_bajo  ic_alto decision
Víspera de festivo         0.934    0.829    1.052  retirar
Se usa el criterio de IC (mas conservador, evita incluir features basadas en ruido de muestra).


In [4]:
incluir_calendario = efectos_calendario.loc[efectos_calendario['decision'] == 'incluir'].sort_values('controlado_x', ascending=False)
retirar_calendario = efectos_calendario.loc[efectos_calendario['decision'] == 'retirar']
cautela_calendario = incluir_calendario.loc[incluir_calendario['sensible_a_especificacion']]

print(f"INCLUIR ({len(incluir_calendario)}): {', '.join(incluir_calendario['evento'])}")
print(f"RETIRAR ({len(retirar_calendario)}): {', '.join(retirar_calendario['evento']) if len(retirar_calendario) else 'ninguno'}")
print()
etiquetas_cautela = [f'{r.evento} ({r.n_anios_evidencia} anios)' for r in cautela_calendario.itertuples()]
print(f'De los incluidos, {len(cautela_calendario)} requieren cautela por evidencia limitada (IC ancho): {etiquetas_cautela}.')

# Se cuenta cuantos de los "cautela" son realmente bloque_diciembre en vez
# de asumirlo -- Semana Santa tambien tiene IC ancho, pero por duracion
# corta del evento (8 dias/anio), no por el hueco de datos de nov-dic 2023.
n_bloque_dic = int((cautela_calendario['tipo'] == 'bloque_diciembre').sum())
bloque_dic_rows = cautela_calendario.loc[cautela_calendario['tipo'] == 'bloque_diciembre']
n_anios_dic = int(bloque_dic_rows['n_anios_evidencia'].max()) if len(bloque_dic_rows) else None
if n_bloque_dic == len(cautela_calendario):
    print(f'Todos ({n_bloque_dic}/{len(cautela_calendario)}) concentran su evidencia en el bloque de diciembre/enero, que',
          f"tiene {n_anios_dic} diciembre(s) completo(s) de evidencia (PARAMS['MESES_2023_INCOMPLETOS'], declarado en 03).")
else:
    otros = cautela_calendario.loc[cautela_calendario['tipo'] != 'bloque_diciembre', 'evento'].tolist()
    print(f'{n_bloque_dic}/{len(cautela_calendario)} concentran su evidencia en el bloque de diciembre/enero',
          f"({n_anios_dic} diciembre(s) completo(s) -- PARAMS['MESES_2023_INCOMPLETOS'], declarado en 03); {otros} es(son) de",
          'evidencia limitada por otro motivo (duracion corta del evento, no por el hueco de datos de 2023).')
incluir_calendario[['evento', 'tipo', 'controlado_x', 'ic_bajo', 'ic_alto', 'n_anios_evidencia', 'sensible_a_especificacion', 'direccion']]


INCLUIR (11): fin_de_anio, nochebuena_navidad, novena, enero_postnavidad, Semana Santa, Puente festivo, Inicio de mes (1-3), Período de prima, Fin de mes (>=28), Quincena (15-17), Festivo
RETIRAR (2): Víspera de festivo, Resaca de festivo

De los incluidos, 5 requieren cautela por evidencia limitada (IC ancho): ['fin_de_anio (3 anios)', 'nochebuena_navidad (3 anios)', 'novena (3 anios)', 'enero_postnavidad (3 anios)', 'Semana Santa (3 anios)'].
4/5 concentran su evidencia en el bloque de diciembre/enero (3 diciembre(s) completo(s) -- PARAMS['MESES_2023_INCOMPLETOS'], declarado en 03); ['Semana Santa'] es(son) de evidencia limitada por otro motivo (duracion corta del evento, no por el hueco de datos de 2023).


,evento,tipo,controlado_x,ic_bajo,ic_alto,n_anios_evidencia,sensible_a_especificacion,direccion
11,fin_de_anio,bloque_diciembre,2.533,2.059,3.115,3,True,sube
10,nochebuena_navidad,bloque_diciembre,1.630,1.320,2.011,3,True,sube
9,novena,bloque_diciembre,1.456,1.309,1.619,3,True,sube
12,enero_postnavidad,bloque_diciembre,1.397,1.227,1.592,3,True,sube
4,Semana Santa,festivo,1.352,1.209,1.512,3,True,sube
3,Puente festivo,festivo,1.342,1.224,1.471,3,False,sube
6,Inicio de mes (1-3),tramo_del_mes,1.317,1.245,1.393,3,False,sube
5,Período de prima,festivo,1.068,1.016,1.123,3,False,sube
8,Fin de mes (>=28),tramo_del_mes,0.929,0.880,0.980,3,False,baja
7,Quincena (15-17),tramo_del_mes,0.908,0.858,0.961,3,False,baja


## 2. Periodicidad semanal y anual

Justificar un termino de Fourier semanal solo con el ACF del notebook
anterior ("confirma periodicidad semanal real") ignoraria que el propio
baseline (seccion 6) muestra que seasonal-naive pierde contra media movil
en todos los origenes a nivel SKU -- el nivel al que la feature va a
operar. Por eso aqui se recalculan ACF y fuerza estacional STL, y se
cruzan con el resultado del baseline antes de justificar nada.

La heterogeneidad categoria x sucursal se contrasta contra una referencia
de ruido (submuestras aleatorias del mismo tamano que la celda mas chica,
tomadas de la celda mas grande) -- el mismo metodo que el notebook anterior
aplico para decidir si la heterogeneidad entre patrones de demanda era
real o tamano de muestra.

La prueba honesta leave-one-year-out se repite aqui sobre unidades
(`panel_diario_negocio`, 01) -- el notebook de calendario la midio sobre
valor; más abajo se reconcilian ambos resultados en vez de dejarlos
sueltos.

In [5]:
panel_negocio = leer_panel_negocio()
serie_negocio_diaria = panel_negocio.groupby('fecha')['cantidad'].sum().reindex(fechas_habiles, fill_value=0)
serie_negocio_diff = serie_negocio_diaria.diff().dropna()
n_serie = len(serie_negocio_diff)

def fuerza_estacional(serie, periodo):
    s = pd.Series(serie).replace(0, np.nan).interpolate().bfill().ffill()
    if s.std() == 0 or len(s) < 2 * periodo:
        return np.nan
    res = STL(s, period=periodo, robust=True).fit()
    var_resid = np.var(res.resid)
    var_total = np.var(res.resid + res.seasonal)
    return max(0, 1 - var_resid / var_total) if var_total > 0 else np.nan

valores_acf = acf_func(serie_negocio_diff, nlags=28, fft=True)
acf_lag7 = float(valores_acf[7])
umbral_significancia_acf = 2 / np.sqrt(n_serie)
fuerza_semanal_negocio = fuerza_estacional(serie_negocio_diaria, PARAMS['VENTANA_ESTACIONAL_DIAS'])

print(f'ACF lag7 sobre la serie diferenciada (unidades, negocio): {acf_lag7:.3f}')
print(f'Umbral de significancia (2/sqrt(n), n={n_serie}): {umbral_significancia_acf:.3f}')
print(f'Fuerza estacional STL semanal (negocio): {fuerza_semanal_negocio:.3f}')


ACF lag7 sobre la serie diferenciada (unidades, negocio): 0.110
Umbral de significancia (2/sqrt(n), n=1059): 0.061
Fuerza estacional STL semanal (negocio): 0.085


El ciclo semanal **es** estadísticamente significativo a nivel
negocio (agregado): el ACF en lag 7 supera el umbral de significancia con
margen. Pero significativo no implica fuerte -- la fuerza estacional STL
es baja-moderada. La sección 7 decide la prioridad de esta feature
cruzando este resultado con el desempeño del baseline a nivel SKU
(sección 6), no con este número aislado.

In [6]:
# Referencia de ruido para la heterogeneidad categoria x sucursal -- grano
# real (SKUs activos dentro de una sola sucursal, no la categoria pooled),
# igual que 04 resampleo pares dentro de un solo patron de demanda.
dispersion_dia_semana = indices_categoria_sucursal.groupby('dia_semana')['indice'].agg(rango=lambda s: s.max() - s.min())
rango_real_max = float(dispersion_dia_semana['rango'].max())

categorias_presentes = indices_categoria_sucursal['categoria'].unique().tolist()
codigos_por_categoria = {cat: set(dim_producto.loc[dim_producto['categoria'] == cat, 'codigo_item']) for cat in categorias_presentes}
tam_celda = {
    (cat, suc): ventas.loc[(ventas['sucursal'] == suc) & (ventas['codigo_item'].isin(codigos_por_categoria[cat])), 'codigo_item'].nunique()
    for cat in categorias_presentes for suc in SUCURSALES_REALES
}
tam_celda_s = pd.Series(tam_celda)
celda_mayor = tam_celda_s.idxmax()
n_min_celda = int(tam_celda_s.min())
cat_mayor, suc_mayor = celda_mayor
codigos_mayor_activos = sorted(ventas.loc[(ventas['sucursal'] == suc_mayor) & (ventas['codigo_item'].isin(codigos_por_categoria[cat_mayor])), 'codigo_item'].unique())

rng = np.random.default_rng(42)
def indice_de_subset(codigos_subset, suc):
    sub = ventas.loc[(ventas['sucursal'] == suc) & (ventas['codigo_item'].isin(codigos_subset))]
    s = sub.groupby('fecha')['valor_total'].sum().reindex(fechas_habiles, fill_value=0)
    return indice_dia_semana(construir_df_razon(s))

indices_submuestras = [
    indice_de_subset([codigos_mayor_activos[j] for j in rng.choice(len(codigos_mayor_activos), size=n_min_celda, replace=False)], suc_mayor)
    for _ in range(5)
]
df_sub = pd.DataFrame(indices_submuestras).T
dispersion_ruido_categoria = float((df_sub.max(axis=1) - df_sub.min(axis=1)).max())
incluir_interaccion_categoria_sucursal = rango_real_max > dispersion_ruido_categoria * 1.5

print(dispersion_dia_semana.round(3))
print()
print(f'celda con menos SKUs activos: {tam_celda_s.idxmin()} ({n_min_celda} SKUs) | celda mas grande usada para el muestreo: {celda_mayor} ({len(codigos_mayor_activos)} SKUs)')
print(f'referencia de ruido (5 submuestras de {n_min_celda} SKUs dentro de {celda_mayor}): {dispersion_ruido_categoria:.3f}')
print(f'rango real observado (maximo entre los 7 dias): {rango_real_max:.3f}')


            rango
dia_semana       
1           0.385
2           0.953
3           0.657
4           0.470
5           0.369
6           0.872
7           0.931

celda con menos SKUs activos: ('Mariscos', 'LA 21') (45 SKUs) | celda mas grande usada para el muestreo: ('Lácteos', 'PRINCIPAL') (239 SKUs)
referencia de ruido (5 submuestras de 45 SKUs dentro de ('Lácteos', 'PRINCIPAL')): 0.261
rango real observado (maximo entre los 7 dias): 0.953


El rango real observado supera ampliamente 1.5 veces la referencia de
ruido -- la heterogeneidad categoría×sucursal es real, no un artefacto del
tamaño de muestra de las celdas más chicas. El catálogo de features debe
incluir la interacción categoría×sucursal para el perfil semanal, no un
solo índice agregado por categoría.

In [7]:
# Leave-one-year-out sobre UNIDADES (esta celda) vs. sobre VALOR (03) -- se
# reconcilian los dos resultados en vez de dejarlos sueltos.
semana_iso = serie_negocio_diaria.index.isocalendar()
serie_negocio_semanal = serie_negocio_diaria.groupby([semana_iso['year'], semana_iso['week']]).sum()

train_loyo = serie_negocio_semanal.loc[serie_negocio_semanal.index.get_level_values(0).isin([2023, 2024])]
test_loyo = serie_negocio_semanal.loc[serie_negocio_semanal.index.get_level_values(0) == 2025]

media_train = train_loyo.mean()
indice_semana_train = train_loyo.groupby(level=1).mean() / media_train
media_test = test_loyo.mean()
semanas_test = test_loyo.index.get_level_values(1)

semanas_sin_dato_train = [int(s) for s in semanas_test if s not in indice_semana_train.index]
pred_estacional = np.array([indice_semana_train.get(s, 1.0) for s in semanas_test]) * media_test
pred_plana = np.full(len(test_loyo), media_test)

mae_estacional = np.abs(test_loyo.values - pred_estacional).mean()
mae_plana = np.abs(test_loyo.values - pred_plana).mean()
incluir_fourier_365 = mae_estacional < mae_plana
mejora_pct = (1 - mae_estacional / mae_plana) * 100

print(f'MAE con indice estacional anual (train 2023-2024, test 2025): {mae_estacional:,.0f}')
print(f'MAE con referencia plana: {mae_plana:,.0f} | mejora: {mejora_pct:.1f}%')
print(f'{len(semanas_sin_dato_train)} semana(s) ISO de 2025 sin dato de entrenamiento (cayeron al fallback de indice=1.0): {semanas_sin_dato_train}')


MAE con indice estacional anual (train 2023-2024, test 2025): 2,704
MAE con referencia plana: 4,382 | mejora: 38.3%
0 semana(s) ISO de 2025 sin dato de entrenamiento (cayeron al fallback de indice=1.0): []


**Caveat de escala**: ambas ramas se escalan por `media_test`, la
media real de 2025 -- esto prueba si la *forma* estacional generaliza dado
el nivel, no es un pronóstico real (que tendría que estimar también el
nivel).

**Reconciliación con el notebook de calendario**: ahí esta misma prueba se
midió sobre valor (COP) y dio ~16.8% de mejora; aquí, sobre unidades (la
escala correcta para una feature de pronóstico de demanda), la mejora es
mayor. Mismo signo -- la señal anual generaliza en ambos casos -- pero
distinta magnitud: el valor mezcla unidades y precio, y el precio tiene su
propia estacionalidad (fin de mes, quincena, sección 1) que infla la
versión medida en valor.

La señal anual **generaliza fuera de muestra en ambas escalas** -- se
incluye el término de Fourier de período 365 en el catálogo, con la
salvedad de que noviembre-diciembre pesan solo 1 año de evidencia (2024)
dentro del entrenamiento (`PARAMS['MESES_2023_INCOMPLETOS']`).

## 3. Precio/promocion y devoluciones -- features de riesgo

Estos dos diagnósticos vivían recalculados aquí (20 líneas duplicadas del
notebook anterior) porque ese notebook no exportaba sus números -- dos
implementaciones del mismo cálculo en la misma cadena. Ahora exporta
`diagnostico_features_riesgo.json`; aquí se lee y se decide, no se
recalcula.

In [8]:
incluir_feature_precio = diagnostico_riesgo['demanda_con_descuento'] > diagnostico_riesgo['demanda_sin_descuento']
print(f"demanda media con proxy de descuento: {diagnostico_riesgo['demanda_con_descuento']:.1f} u/dia | sin descuento: {diagnostico_riesgo['demanda_sin_descuento']:.1f} u/dia")

incluir_feature_devolucion = diagnostico_riesgo['pct_devoluciones_perecedero_refrigerado'] > diagnostico_riesgo['pct_catalogo_perecedero_refrigerado']
print(f"{diagnostico_riesgo['pct_devoluciones_perecedero_refrigerado']:.1%} de las devoluciones son de productos perecederos/refrigerados vs.",
      f"{diagnostico_riesgo['pct_catalogo_perecedero_refrigerado']:.1%} de esas categorias en el catalogo.")


demanda media con proxy de descuento: 43.5 u/dia | sin descuento: 46.3 u/dia
18.8% de las devoluciones son de productos perecederos/refrigerados vs. 11.8% de esas categorias en el catalogo.


**Precio**: se retira de la lista de features de prioridad alta -- el
proxy de descuento vende *menos*, efecto contrario al esperado, así que
probablemente mezcla cambios de referencia/proveedor y no una promoción
real. Queda como feature exploratoria de baja prioridad.

**Devoluciones**: se incluye la tasa de devolución por categoría como
feature de riesgo de calidad para el clasificador de nivel 2 -- se
concentran en perecederos/refrigerados por encima de su peso en el
catálogo.

## 4. Producto, patron de demanda y racha -- enrutamiento y riesgo de quiebre

`racha_por_par.parquet` (exportado por el notebook anterior) permite leer
y reverificar aquí directamente el hallazgo sobre racha máxima de ceros --
si la racha mayor de los productos prioritarios es solo composición por
perecederos/temporada, o evidencia directa de desabastecimiento.

In [9]:
tabla_patron_diario = pd.crosstab(patron_demanda['patron_diario'], patron_demanda['es_prioritario'], normalize='columns') * 100
tabla_patron_semanal = pd.crosstab(patron_demanda['patron_semanal'], patron_demanda['es_prioritario'], normalize='columns') * 100

pct_intermitente_diario = float(tabla_patron_diario.loc['intermitente', True]) if 'intermitente' in tabla_patron_diario.index else 0.0
pct_suave_diario = float(tabla_patron_diario.loc['suave', True]) if 'suave' in tabla_patron_diario.index else 0.0
pct_suave_semanal = float(tabla_patron_semanal.loc['suave', True]) if 'suave' in tabla_patron_semanal.index else 0.0
mayoria_intermitente = pct_intermitente_diario > 50

print('Patron diario (% del subconjunto prioritario):')
print(tabla_patron_diario[True].round(1))
print()
print('Patron semanal (% del subconjunto prioritario):')
print(tabla_patron_semanal[True].round(1))


Patron diario (% del subconjunto prioritario):
patron_diario
erratico        4.600
intermitente   71.100
lumpy          18.000
sin_datos       3.000
suave           3.300
Name: True, dtype: float64

Patron semanal (% del subconjunto prioritario):
patron_semanal
erratico        4.700
intermitente   52.700
lumpy          11.500
sin_datos       3.400
suave          27.600
Name: True, dtype: float64


El subconjunto priorizado está **dominado por demanda intermitente**
a grano diario (suave es minoría); a grano semanal "suave" sube pero sigue
sin ser mayoría. El enrutamiento de modelo (sección 8) no puede asumir una
sola familia de modelo para todo el conjunto.

In [10]:
alta_frecuencia_racha = racha_por_par.loc[racha_por_par['frecuencia_venta'] >= PARAMS['UMBRAL_FRECUENCIA_RACHA_CONTROL']]
tabla_racha_grupo = alta_frecuencia_racha.groupby('grupo_condicion')['racha_max_ceros'].describe()[['count', 'mean', '50%']]
print(tabla_racha_grupo)

mediana_6_7 = tabla_racha_grupo.loc['solo_6_o_7', '50%'] if 'solo_6_o_7' in tabla_racha_grupo.index else np.nan
mediana_no_prior = tabla_racha_grupo.loc['no_prioritario', '50%']
racha_es_evidencia_directa = pd.notna(mediana_6_7) and mediana_6_7 > mediana_no_prior


                  count  mean   50%
grupo_condicion                    
ambas            76.000 3.434 3.000
no_prioritario  652.000 6.161 0.000
solo_1_a_5      193.000 7.249 5.000
solo_6_o_7      200.000 8.485 6.000


Releído directamente de `racha_por_par.parquet`: el grupo
`solo_6_o_7` (volumen/rotación, sin razón estructural para rachas largas)
tiene racha mayor que los no-prioritarios -- no es solo composición por
perecederos/temporada. Es evidencia directa de que el problema de
desabastecimiento se concentra en los productos que al negocio le importa
reponer.

## 5. Contrafactual ABC -- cuanto valor se sacrifica por priorizar riesgo operativo

El notebook de la regla de priorización calculó este contrafactual, pero
el número nunca llegó a la síntesis. Se recalcula aquí sobre
`fact_ventas.csv` (mismo método: mismo tamaño de catálogo, top-N por
valor) y se agrega el número que faltaba -- cuánto valor aportan,
exclusivamente, los productos que la regla prioriza y el ABC no.

In [11]:
n_equivalente = len(prioritarios)
top_valor = valor_por_producto.sort_values(ascending=False).head(n_equivalente)
codigos_abc = set(top_valor.index)
valor_abc_pct = top_valor.sum() / valor_total
pct_valor_regla = valor_por_producto.reindex(prioritarios).sum() / valor_total

interseccion = codigos_abc & prioritarios
solo_regla = prioritarios - codigos_abc
valor_interseccion_pct = valor_por_producto.reindex(interseccion).sum() / valor_total
valor_solo_regla_pct = valor_por_producto.reindex(solo_regla).sum() / valor_total

print(f'Regla de negocio: {pct_valor_regla:.1%} del valor historico con {n_equivalente} productos.')
print(f'ABC puro (mismo tamano de catalogo): {valor_abc_pct:.1%} del valor historico.')
print(f'Interseccion: {len(interseccion)} productos ({valor_interseccion_pct:.1%} del valor historico).')
print(f'Solo la regla (el ABC no los tomaria): {len(solo_regla)} productos, que aportan apenas {valor_solo_regla_pct:.1%}',
      'del valor historico.')


Regla de negocio: 51.4% del valor historico con 919 productos.
ABC puro (mismo tamano de catalogo): 79.4% del valor historico.
Interseccion: 349 productos (47.0% del valor historico).
Solo la regla (el ABC no los tomaria): 570 productos, que aportan apenas 4.4% del valor historico.


En valor, la regla de negocio es un **ABC parcial**: una parte
sustancial del valor que captura ya la captura un ABC puro solo. Los
productos exclusivos de la regla (que el ABC no tomaría) mueven muy poco
ingreso -- confirma que sus criterios (perecederos, espacio en bodega,
temporada) son de riesgo operativo, no de maximización de ingreso. Un
modelo que optimice solo WAPE o valor esperado subestimaría
sistemáticamente la prioridad de negocio de ese subconjunto.

## 6. Baseline: WAPE por origen/patron y piso de comparacion

Se exporta `piso_baseline_por_patron.csv` -- el número más accionable de
toda la cadena para el siguiente paso del proyecto (el WAPE que cualquier
modelo candidato debe superar) -- y se cita explícitamente en la
recomendación de la sección 8, no solo se describe.

In [12]:
BASELINES = ['naive', 'estacional', 'media_movil']

def wape_agregado(grupo, col_err):
    return grupo[col_err].sum() / grupo['actual_sum'].sum()

def tabla_wape_por(columna_grupo):
    return baseline_wape.groupby(columna_grupo).apply(
        lambda g: pd.Series({
            f'wape_{b}_{vista}': wape_agregado(g, f'abs_err_{b}_{vista}')
            for b in BASELINES for vista in ('diario', 'horizonte')
        }), include_groups=False
    ).round(3)

resumen_por_origen = tabla_wape_por('origen')
peor_origen = resumen_por_origen['wape_naive_diario'].idxmax()
mejor_origen = resumen_por_origen['wape_naive_diario'].idxmin()
gana_estacional = (resumen_por_origen['wape_estacional_diario'] < resumen_por_origen['wape_media_movil_diario'])
piso_es_media_movil = not gana_estacional.all()

print(resumen_por_origen[['wape_naive_diario', 'wape_estacional_diario', 'wape_media_movil_diario']])
print()
print(f'peor origen: {peor_origen} | mejor origen: {mejor_origen}')
print(f'seasonal-naive le gana a media movil en {gana_estacional.sum()}/{len(gana_estacional)} origenes.')


                wape_naive_diario  wape_estacional_diario  \
origen                                                      
diciembre_alto              1.003                   0.984   
ordinario_1                 1.017                   0.987   
ordinario_2                 1.034                   1.052   
semana_santa                1.108                   0.996   

                wape_media_movil_diario  
origen                                   
diciembre_alto                    0.898  
ordinario_1                       0.887  
ordinario_2                       0.944  
semana_santa                      0.888  

peor origen: semana_santa | mejor origen: diciembre_alto
seasonal-naive le gana a media movil en 0/4 origenes.


Retomando la sección 2: el ciclo semanal es significativo a nivel
negocio (agregado), pero seasonal-naive pierde contra media móvil en
**todos** los orígenes a nivel SKU (arriba). La periodicidad semanal
existe en el agregado, pero no es explotable por un método tan simple a
nivel de SKU individual -- consistente con la fuerza STL baja-moderada
calculada en la sección 2.

In [13]:
# 'sin_datos' (pares casi sin venta, ver 04 seccion 1) tiene actual_sum=0 en
# algunos casos -- se excluye del piso, igual que 04 lo excluyo del indice
# estacional por patron.
piso_baseline_por_patron = baseline_wape.loc[baseline_wape['patron'] != 'sin_datos'].groupby('patron').apply(
    lambda g: pd.Series({
        'wape_piso_diario': g['abs_err_media_movil_diario'].sum() / g['actual_sum'].sum(),
        'wape_piso_horizonte': g['abs_err_media_movil_horizonte'].sum() / g['actual_sum'].sum(),
    }), include_groups=False
).round(3)
piso_baseline_por_patron.to_csv(f'{DW}/piso_baseline_por_patron.csv')
print('guardado:', f'{DW}/piso_baseline_por_patron.csv')
piso_baseline_por_patron


guardado: /home/ddelgadillo@redcorporativa.corpoica.org.co/otros/etl/InventaIO/data/processed_real/piso_baseline_por_patron.csv


,wape_piso_diario,wape_piso_horizonte
patron,,
erratico,0.686,0.360
intermitente,1.297,0.700
lumpy,1.091,0.572
suave,0.445,0.199


## 7. Catalogo consolidado de features (`features_recomendadas.csv`)

La columna `origen_decision` distingue `evidencia` (la inclusión/exclusión
depende de un número calculado en este notebook o en el de calendario/
demanda) de `estandar` (features que entran por diseño -- lags, atributos
de producto -- sin necesitar una prueba estadística).

`decision` (incluir/retirar), `nivel` (1/2/enrutamiento) y `prioridad`
(alta/baja) van en columnas separadas -- un filtro `decision == 'incluir'`
no pierde las features de nivel 2 ni arrastra la de enrutamiento por
error.

In [14]:
filas = []

for r in efectos_calendario.itertuples():
    justificacion = f'{r.direccion}, {r.controlado_x:.2f}x IC[{r.ic_bajo:.2f},{r.ic_alto:.2f}], {r.n_anios_evidencia} anios de evidencia'
    if r.decision == 'incluir' and r.sensible_a_especificacion:
        justificacion += ' -- INCLUIR CON CAUTELA (IC ancho, evidencia limitada)'
    filas.append({
        'feature': f'evento_calendario__{r.evento}', 'grano': 'dia', 'fuente': 'efectos_calendario.csv (03)',
        'decision': r.decision, 'nivel': '1', 'prioridad': 'alta', 'origen_decision': 'evidencia', 'justificacion': justificacion,
    })

texto_interaccion = 'categoria x sucursal' if incluir_interaccion_categoria_sucursal else 'categoria'
justificacion_interaccion = (
    f'rango real {rango_real_max:.2f} vs. referencia de ruido {dispersion_ruido_categoria:.2f} (celda {celda_mayor}, n={n_min_celda}) -- '
    + ('supera 1.5x el ruido, interaccion justificada' if incluir_interaccion_categoria_sucursal else 'del orden del ruido, agregado por categoria basta')
)
justificacion_fourier7 = (
    f'ACF lag7={acf_lag7:.2f} (significativo, umbral 2/sqrt(n)={umbral_significancia_acf:.2f}) y fuerza STL semanal={fuerza_semanal_negocio:.2f} '
    f'a nivel NEGOCIO, pero seasonal-naive pierde contra media movil en {(~gana_estacional).sum()}/{len(gana_estacional)} origenes a nivel SKU '
    '-- periodicidad real mas debil a nivel SKU que a nivel agregado.'
)
justificacion_fourier28 = 'patron ACF no plano (lag28 dos veces lag7, notebook de calendario) sugiere ciclo mensual adicional -- misma cautela que fourier_periodo_7 (senal agregada mas fuerte que a nivel SKU), sin prueba de significancia propia.'
justificacion_fourier365 = f'MAE estacional {mae_estacional:,.0f} vs. MAE plano {mae_plana:,.0f} en test 2025 (mejora {mejora_pct:.1f}%), leave-one-year-out 2023-2024 -> 2025, sobre unidades.'
justificacion_precio = f"demanda media con proxy de descuento {diagnostico_riesgo['demanda_con_descuento']:.1f} u/dia vs. {diagnostico_riesgo['demanda_sin_descuento']:.1f} sin descuento (04)."
justificacion_devolucion = f"{diagnostico_riesgo['pct_devoluciones_perecedero_refrigerado']:.1%} de devoluciones en perecederos/refrigerados vs. {diagnostico_riesgo['pct_catalogo_perecedero_refrigerado']:.1%} del catalogo (04)."
justificacion_racha = (
    f'racha_por_par.parquet (04): mediana solo_6_o_7={mediana_6_7:.0f} vs. no_prioritario={mediana_no_prior:.0f} -- '
    + ('evidencia directa de desabastecimiento, no solo composicion' if racha_es_evidencia_directa else 'principalmente composicion por perecederos/temporada')
)

filas += [
    dict(feature='lags_1_7_14_28_y_medias_moviles', grano='producto x sucursal x dia', fuente='panel_diario.parquet (01)',
         decision='incluir', nivel='1', prioridad='alta', origen_decision='estandar',
         justificacion='estandar; en patrones intermitente/lumpy (ADI alto) la mayoria de lags son cero -- complementa a Croston/TSB, no lo reemplaza.'),
    dict(feature='dias_desde_ultima_venta', grano='producto x sucursal x dia', fuente='panel_diario.parquet + ventana_activa.parquet (01)',
         decision='incluir', nivel='2', prioridad='alta', origen_decision='estandar',
         justificacion='proxy de riesgo de quiebre (04, seccion 2) -- racha maxima, con censura por la derecha declarada en 04.'),
    dict(feature='frecuencia_venta_historica', grano='producto x sucursal', fuente='ventana_activa.parquet (01)',
         decision='incluir', nivel='1', prioridad='alta', origen_decision='estandar', justificacion='feature continua de rotacion, ya usada para filtrar la seccion 4 de 04.'),
    dict(feature='fourier_periodo_7', grano='dia', fuente='esta seccion 2 (ACF+STL) cruzado con baseline (seccion 6)',
         decision='incluir', nivel='1', prioridad='baja', origen_decision='evidencia', justificacion=justificacion_fourier7),
    dict(feature='fourier_periodo_28_30', grano='dia', fuente='03, seccion 6 (hallazgo B2 de ACF)',
         decision='incluir', nivel='1', prioridad='baja', origen_decision='evidencia', justificacion=justificacion_fourier28),
    dict(feature='fourier_periodo_365', grano='dia', fuente='leave-one-year-out (seccion 2 de este notebook)',
         decision=('incluir' if incluir_fourier_365 else 'retirar'), nivel='1', prioridad='alta', origen_decision='evidencia', justificacion=justificacion_fourier365),
    dict(feature=f'indice_estacional_{texto_interaccion.replace(" ", "_")}', grano=texto_interaccion,
         fuente='estacionalidad_indices_categoria_sucursal.csv (03)', decision='incluir', nivel='1', prioridad='alta',
         origen_decision='evidencia', justificacion=justificacion_interaccion),
    dict(feature='categoria_perecedero_refrigerado_volumen_peso', grano='producto', fuente='dim_producto_priorizado.parquet (02)',
         decision='incluir', nivel='1', prioridad='alta', origen_decision='estandar',
         justificacion='atributos de producto y las 7 banderas de la regla de priorizacion, explicables por diseno.'),
    dict(feature='patron_adi_cv2_diario_y_semanal', grano='producto x sucursal', fuente='patron_demanda_producto_sucursal.parquet (04)',
         decision='incluir', nivel='enrutamiento', prioridad='alta', origen_decision='estandar',
         justificacion='no es feature de entrada al modelo de demanda -- es la variable que decide que familia de modelo usar (seccion 8).'),
    dict(feature='precio_proxy_descuento', grano='producto x dia', fuente='diagnostico_features_riesgo.json (04)',
         decision=('incluir' if incluir_feature_precio else 'retirar'), nivel='1',
         prioridad=('alta' if incluir_feature_precio else 'baja'), origen_decision='evidencia', justificacion=justificacion_precio),
    dict(feature='tasa_devolucion_categoria', grano='categoria', fuente='diagnostico_features_riesgo.json (04)',
         decision=('incluir' if incluir_feature_devolucion else 'retirar'), nivel='2', prioridad='alta',
         origen_decision='evidencia', justificacion=justificacion_devolucion),
    dict(feature='dias_cobertura', grano='producto x sucursal x dia', fuente='fact_inventario.csv',
         decision=('incluir' if 'dias_cobertura' in fact_inventario_cols else 'retirar'), nivel='2', prioridad='alta',
         origen_decision='estandar', justificacion='insumo directo del clasificador de riesgo de quiebre (nivel 2).'),
    dict(feature='lead_time_dias', grano='proveedor', fuente='dim_proveedor.csv',
         decision=('incluir' if 'lead_time_dias' in dim_proveedor.columns else 'retirar'), nivel='2', prioridad='alta',
         origen_decision='estandar', justificacion='insumo directo del clasificador de riesgo de quiebre (nivel 2).'),
    dict(feature='racha_max_ceros_alta_frecuencia', grano='producto x sucursal', fuente='racha_por_par.parquet (04)',
         decision='incluir', nivel='2', prioridad='alta', origen_decision='evidencia', justificacion=justificacion_racha),
]

features_recomendadas = pd.DataFrame(filas)
assert set(features_recomendadas['decision']) <= {'incluir', 'retirar'}, 'vocabulario de decision fuera de lo esperado'
assert set(features_recomendadas['nivel']) <= {'1', '2', 'enrutamiento'}, 'vocabulario de nivel fuera de lo esperado'
assert set(features_recomendadas['prioridad']) <= {'alta', 'baja'}, 'vocabulario de prioridad fuera de lo esperado'

features_recomendadas.to_csv(f'{DW}/features_recomendadas.csv', index=False)
print('guardado:', f'{DW}/features_recomendadas.csv', features_recomendadas.shape)
print(features_recomendadas.groupby(['decision', 'nivel', 'prioridad', 'origen_decision']).size())
features_recomendadas


guardado: /home/ddelgadillo@redcorporativa.corpoica.org.co/otros/etl/InventaIO/data/processed_real/features_recomendadas.csv (27, 8)
decision  nivel         prioridad  origen_decision
incluir   1             alta       estandar            3
                                   evidencia          13
                        baja       evidencia           2
          2             alta       estandar            3
                                   evidencia           2
          enrutamiento  alta       estandar            1
retirar   1             alta       evidencia           2
                        baja       evidencia           1
dtype: int64


,feature,grano,fuente,decision,nivel,prioridad,origen_decision,justificacion
0,evento_calendario__Festivo,dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"baja, 0.87x IC[0.80,0.95], 3 anios de evidencia"
1,evento_calendario__Víspera de festivo,dia,efectos_calendario.csv (03),retirar,1,alta,evidencia,"sin_efecto_claro (IC cruza 1.0), 0.93x IC[0.83..."
2,evento_calendario__Resaca de festivo,dia,efectos_calendario.csv (03),retirar,1,alta,evidencia,"sin_efecto_claro (IC cruza 1.0), 1.04x IC[0.96..."
3,evento_calendario__Puente festivo,dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"sube, 1.34x IC[1.22,1.47], 3 anios de evidencia"
4,evento_calendario__Semana Santa,dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"sube, 1.35x IC[1.21,1.51], 3 anios de evidenci..."
5,evento_calendario__Período de prima,dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"sube, 1.07x IC[1.02,1.12], 3 anios de evidencia"
6,evento_calendario__Inicio de mes (1-3),dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"sube, 1.32x IC[1.25,1.39], 3 anios de evidencia"
7,evento_calendario__Quincena (15-17),dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"baja, 0.91x IC[0.86,0.96], 3 anios de evidencia"
8,evento_calendario__Fin de mes (>=28),dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"baja, 0.93x IC[0.88,0.98], 3 anios de evidencia"
9,evento_calendario__novena,dia,efectos_calendario.csv (03),incluir,1,alta,evidencia,"sube, 1.46x IC[1.31,1.62], 3 anios de evidenci..."


## 8. Recomendacion de enfoque de modelado

Generada a partir de las variables calculadas en las secciones 1-7 -- no
escrita a mano.

In [15]:
texto_intermitencia = 'esta DOMINADO por demanda intermitente' if mayoria_intermitente else 'NO esta dominado por un solo patron'
texto_gana = 'le gana' if gana_estacional.all() else ('NO le gana' if not gana_estacional.any() else 'le gana solo en algunos origenes')
texto_piso = 'media movil' if piso_es_media_movil else 'seasonal-naive'
texto_cobertura = 'disponible' if 'dias_cobertura' in fact_inventario_cols else 'NO disponible'
texto_lead_time = 'disponible' if 'lead_time_dias' in dim_proveedor.columns else 'NO disponible'
texto_devolucion = 'senal confirmada' if incluir_feature_devolucion else 'senal NO confirmada'
fila_semana_santa = efectos_calendario.set_index('evento').loc['Semana Santa'] if 'Semana Santa' in efectos_calendario['evento'].values else None
piso_del_peor_origen = resumen_por_origen.loc[peor_origen, 'wape_media_movil_diario'] if piso_es_media_movil else resumen_por_origen.loc[peor_origen, 'wape_estacional_diario']
piso_intermitente = piso_baseline_por_patron.loc['intermitente', 'wape_piso_diario'] if 'intermitente' in piso_baseline_por_patron.index else np.nan
piso_suave = piso_baseline_por_patron.loc['suave', 'wape_piso_diario'] if 'suave' in piso_baseline_por_patron.index else np.nan

lineas = []
lineas.append('NIVEL 1 -- enrutamiento por patron de demanda (ADI/CV2, seccion 4).')
lineas.append(f'El subconjunto priorizado {texto_intermitencia} a grano diario ({pct_intermitente_diario:.1f}% intermitente, {pct_suave_diario:.1f}% suave); a grano semanal "suave" sube a {pct_suave_semanal:.1f}%.')
lineas.append('')
lineas.append(f'El baseline (04) muestra que el peor origen de pronostico es {peor_origen} (piso WAPE {piso_del_peor_origen:.3f}) y el mejor {mejor_origen} -- un modelo validado solo en meses ordinarios se cae justo donde mas le importa al negocio.')
if fila_semana_santa is not None and peor_origen == 'semana_santa':
    lineas.append(f'Coincide con el efecto de calendario mas fuerte fuera de diciembre medido en 03 (Semana Santa {fila_semana_santa["controlado_x"]:.2f}x, IC [{fila_semana_santa["ic_bajo"]:.2f},{fila_semana_santa["ic_alto"]:.2f}]) -- confirma que el modelo necesita esa feature explicita, no solo dia de semana + tendencia.')
lineas.append('')
lineas.append(f'seasonal-naive {texto_gana} a media movil ({gana_estacional.sum()}/{len(gana_estacional)} origenes) -- el piso de comparacion para cualquier modelo candidato es {texto_piso} (exportado en piso_baseline_por_patron.csv: {piso_suave:.3f} para suave, {piso_intermitente:.3f} para intermitente, vista diaria).')
lineas.append('')
lineas.append('Familia de modelo por patron:')
lineas.append(f'- Demanda SUAVE: modelo global de gradient boosting (LightGBM/XGBoost) con las features de la seccion 7; debe superar el WAPE piso de {piso_suave:.3f} (patron suave, vista diaria) y en particular el de {peor_origen} ({piso_del_peor_origen:.3f}), no solo el promedio.')
lineas.append(f'- Demanda intermitente/erratica/lumpy (mayoria del conjunto priorizado, piso {piso_intermitente:.3f}): Croston/SBA o TSB como linea base de primera linea, con el indice estacional de {texto_interaccion} (seccion 2) como feature en vez de lags cortos.')
lineas.append('')
lineas.append(f'Contrafactual (seccion 5): la regla de negocio captura {pct_valor_regla:.1%} del valor historico; los {len(solo_regla)} productos que solo la regla prioriza (no un ABC puro) aportan apenas {valor_solo_regla_pct:.1%} -- el modelo de nivel 1 no debe optimizarse solo por WAPE/valor esperado, perderia de vista ese subconjunto de riesgo operativo.')
lineas.append('')
lineas.append('NIVEL 2 -- clasificador de riesgo de quiebre:')
lineas.append(f'P(stock < demanda proyectada 7/15 dias), usando dias_cobertura ({texto_cobertura}), lead_time_dias ({texto_lead_time}), racha_max_ceros_alta_frecuencia (racha_por_par.parquet, 04), y tasa de devolucion por categoria ({texto_devolucion}).')
lineas.append('')
lineas.append('Metrica: pinball loss en cuantiles altos (0.8-0.95) para el pronostico de demanda; WAPE (diario y de ventana acumulada) contra el piso de piso_baseline_por_patron.csv, no contra el promedio general.')
lineas.append('Validacion: walk-forward con folds que incluyan explicitamente diciembre Y Semana Santa -- nunca solo meses ordinarios, el sesgo que el propio baseline de la seccion 6 expone.')

print(chr(10).join(lineas))


NIVEL 1 -- enrutamiento por patron de demanda (ADI/CV2, seccion 4).
El subconjunto priorizado esta DOMINADO por demanda intermitente a grano diario (71.1% intermitente, 3.3% suave); a grano semanal "suave" sube a 27.6%.

El baseline (04) muestra que el peor origen de pronostico es semana_santa (piso WAPE 0.888) y el mejor diciembre_alto -- un modelo validado solo en meses ordinarios se cae justo donde mas le importa al negocio.
Coincide con el efecto de calendario mas fuerte fuera de diciembre medido en 03 (Semana Santa 1.35x, IC [1.21,1.51]) -- confirma que el modelo necesita esa feature explicita, no solo dia de semana + tendencia.

seasonal-naive NO le gana a media movil (0/4 origenes) -- el piso de comparacion para cualquier modelo candidato es media movil (exportado en piso_baseline_por_patron.csv: 0.445 para suave, 1.297 para intermitente, vista diaria).

Familia de modelo por patron:
- Demanda SUAVE: modelo global de gradient boosting (LightGBM/XGBoost) con las features de la se

## 9. Proximos pasos

1. Confirmar con el negocio el umbral de frecuencia de la condicion 7
   (`PARAMS['UMBRAL_FRECUENCIA_COND7']`, ver 02) como reemplazo del "top 50"
   fijo original.
2. Construir el dataset de entrenamiento sobre
   `priorizacion_producto_sucursal.parquet` (grano correcto, 02) primero
   para el subconjunto prioritario, incorporando `features_recomendadas.csv`
   de la seccion 7 como contrato de features.
3. Prototipar el enrutamiento ADI/CV2 -> familia de modelo con backtesting
   real sobre los 4 origenes de `PARAMS['ORIGENES_BASELINE']`, comparando
   contra `piso_baseline_por_patron.csv` (04/05) por origen y por patron --
   no un fold unico.
4. Revisar con el negocio los falsos positivos/negativos de la condicion 5
   ya identificados en 03 antes de dar la regla de priorizacion por cerrada.
5. Decidir grano diario vs. semanal por PATRON de demanda (seccion 4), no
   una eleccion unica para todo el conjunto priorizado.
6. Si se levanta la restriccion de datos de `PARAMS['MESES_2023_INCOMPLETOS']`
   (ej. si aparecen los reportes faltantes de nov-dic 2023), reejecutar 03,
   04 y 05 -- varias decisiones de esta sintesis (fin_de_anio, novena,
   Fourier-365) dependen de una muestra de 2 diciembres, no 3.
7. Renombrar la columna `sensible_a_especificacion` de
   `efectos_calendario.csv` -- hoy mide ancho de intervalo de confianza, no
   sensibilidad a la especificación del modelo; el nombre induce a error
   aunque el texto de este notebook la interprete correctamente.
8. Recuperar una prueba real de sensibilidad a la especificación (comparar
   el efecto estimado bajo formulaciones alternativas del modelo conjunto
   de calendario) -- se perdió al migrar del control por media móvil al
   modelo de regresión conjunta, y mide algo distinto de lo que el ancho
   del intervalo de confianza ya captura.

## 10. Verificacion y huella

In [16]:
assert len(features_recomendadas) > 0, 'catalogo vacio'
print('Invariantes verificadas OK.')

huella = {
    'notebook': '05_sintesis',
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'entradas': registrar_huella([
        'dim_producto_priorizado.parquet', 'priorizacion_producto_sucursal.parquet',
        'efectos_calendario.csv', 'estacionalidad_sku.parquet', 'estacionalidad_indices_categoria_sucursal.csv',
        'patron_demanda_producto_sucursal.parquet', 'baseline_wape.parquet',
        'racha_por_par.parquet', 'diagnostico_features_riesgo.json',
        'dim_proveedor.csv', 'fact_inventario.csv', 'fact_ventas.csv',
    ]),
    'salidas': registrar_huella(['features_recomendadas.csv', 'piso_baseline_por_patron.csv']),
}
with open(f'{DW}/huella_05.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
print('guardado: huella_05.json')
huella


Invariantes verificadas OK.


guardado: huella_05.json


{'notebook': '05_sintesis',
 'fecha_generacion': '2026-09-18T09:42:33.315980',
 'entradas': {'dim_producto_priorizado.parquet': '68e0a2fc304c',
  'priorizacion_producto_sucursal.parquet': '0750d5792cc8',
  'efectos_calendario.csv': '32345cb1990a',
  'estacionalidad_sku.parquet': '2b0d29a0dd73',
  'estacionalidad_indices_categoria_sucursal.csv': '7de2c7d4d6ff',
  'patron_demanda_producto_sucursal.parquet': '843a35944f56',
  'baseline_wape.parquet': '275719867517',
  'racha_por_par.parquet': '95e8e2bb31d3',
  'diagnostico_features_riesgo.json': 'fcc0f35cc279',
  'dim_proveedor.csv': '589ae5214b01',
  'fact_inventario.csv': 'b6299786f178',
  'fact_ventas.csv': '904aaeef7217'},
 'salidas': {'features_recomendadas.csv': '4941898e8398',
  'piso_baseline_por_patron.csv': '8aa20b713fd5'}}